# Pan-UK Biobank: a multi-million-variant Manhattan plot

Each Pan-UKBB per-phenotype file contains 28,987,534 variants. This
notebook reads indexed windows from the standing-height GWAS and plots
the released `−log10(p)` statistic across all autosomes. The full
2.0 GB flat file stays remote; only its 2 MB tabix index and requested
BGZF blocks are transferred.

Increase `PANUKBB_WINDOWS` or `PANUKBB_VARIANTS_PER_WINDOW` for a
denser run.

**Source:** [Pan-UKBB downloads](https://pan.ukbb.broadinstitute.org/downloads/index.html)
and [per-phenotype file documentation](https://pan.ukbb.broadinstitute.org/docs/per-phenotype-files/index.html).
The data are CC BY 4.0; publications should acknowledge Pan-UKBB and
UK Biobank as requested on the download page.

Install beside XY with `python -m pip install numpy pysam requests xy`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pysam
import requests

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = (
    "https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/"
    "continuous-50-both_sexes-irnt.tsv.bgz"
)
INDEX_URL = (
    "https://pan-ukb-us-east-1.s3.amazonaws.com/"
    "sumstats_flat_files_tabix/"
    "continuous-50-both_sexes-irnt.tsv.bgz.tbi"
)
index_path = DATA_DIR / "continuous-50-both_sexes-irnt.tsv.bgz.tbi"
if not index_path.exists():
    response = requests.get(INDEX_URL, timeout=120)
    response.raise_for_status()
    index_path.write_bytes(response.content)

CHROMOSOME_LENGTHS = {
    1: 249_250_621,
    2: 243_199_373,
    3: 198_022_430,
    4: 191_154_276,
    5: 180_915_260,
    6: 171_115_067,
    7: 159_138_663,
    8: 146_364_022,
    9: 141_213_431,
    10: 135_534_747,
    11: 135_006_516,
    12: 133_851_895,
    13: 115_169_878,
    14: 107_349_540,
    15: 102_531_392,
    16: 90_354_753,
    17: 81_195_210,
    18: 78_077_248,
    19: 59_128_983,
    20: 63_025_520,
    21: 48_129_895,
    22: 51_304_566,
}
windows_per_chromosome = int(os.getenv("PANUKBB_WINDOWS", "8"))
variants_per_window = int(os.getenv("PANUKBB_VARIANTS_PER_WINDOW", "15000"))
window_width = int(os.getenv("PANUKBB_WINDOW_BP", "3000000"))
if min(windows_per_chromosome, variants_per_window, window_width) <= 0:
    raise ValueError("Pan-UKBB window controls must be positive")

In [ ]:
offsets = {}
running_offset = 0
for chromosome, length in CHROMOSOME_LENGTHS.items():
    offsets[chromosome] = running_offset
    running_offset += length

position_parts = []
significance_parts = []
chromosome_parts = []

# The immutable quantitative-trait schema starts with:
# chr, pos, ref, alt, af_meta_hq, beta_meta_hq, se_meta_hq,
# neglog10_pval_meta_hq. Its plain TSV header is skipped by tabix.
position_column = 1
pvalue_column = 7

with pysam.TabixFile(DATA_URL, index=str(index_path)) as summary:
    for chromosome, length in CHROMOSOME_LENGTHS.items():
        centers = (
            (np.arange(windows_per_chromosome, dtype=np.float64) + 0.5)
            * length
            / windows_per_chromosome
        )
        starts = np.clip(
            centers - window_width / 2,
            0,
            max(0, length - window_width),
        ).astype(np.int64)
        positions = []
        significance = []
        for start in starts:
            kept = 0
            records = summary.fetch(
                str(chromosome),
                int(start),
                int(start + window_width),
            )
            for line in records:
                fields = line.split("\t")
                value = fields[pvalue_column]
                if value == "NA":
                    continue
                positions.append(offsets[chromosome] + int(fields[position_column]))
                significance.append(float(value))
                kept += 1
                if kept >= variants_per_window:
                    break

        position_parts.append(np.asarray(positions, dtype=np.float64))
        significance_parts.append(np.asarray(significance, dtype=np.float64))
        chromosome_parts.append(np.full(len(positions), chromosome, dtype=np.float64))
        print(f"chr{chromosome}: {len(positions):,} variants")

genomic_position = np.concatenate(position_parts)
neglog10_pvalue = np.concatenate(significance_parts)
chromosome_number = np.concatenate(chromosome_parts)
print(f"{genomic_position.size:,} variants total")

In [ ]:
tick_values = [
    offsets[chromosome] + CHROMOSOME_LENGTHS[chromosome] / 2 for chromosome in CHROMOSOME_LENGTHS
]
tick_labels = [str(chromosome) for chromosome in CHROMOSOME_LENGTHS]
chromosome_boundaries = [offsets[chromosome] for chromosome in range(2, 23)]

# Violet and coral distinguish adjacent chromosomes in the compact hero.
aubergine = np.array([124, 58, 237, 255], dtype=np.float32) / 255
ochre = np.array([251, 113, 133, 255], dtype=np.float32) / 255
point_rgba = np.empty((chromosome_number.size, 4), dtype=np.float32)
odd_chromosome = chromosome_number % 2 == 1
point_rgba[odd_chromosome] = aubergine
point_rgba[~odd_chromosome] = ochre

chr20_indices = np.flatnonzero(chromosome_number == 20)
chr20_peak_index = int(chr20_indices[np.argmax(neglog10_pvalue[chr20_indices])])
other_peak_index = int(np.argmax(np.where(chromosome_number == 20, -np.inf, neglog10_pvalue)))
display_y_max = 180.0
off_scale_y = 172.0
peak_specs = (
    (chr20_peak_index, -14, 18, "end"),
    (other_peak_index, 14, -24, "start"),
)
peak_labels = []
for peak_index, _, _, _ in peak_specs:
    chromosome = int(chromosome_number[peak_index])
    position = int(genomic_position[peak_index] - offsets[chromosome])
    value = float(neglog10_pvalue[peak_index])
    suffix = " · OFF-SCALE" if value > display_y_max else ""
    peak_labels.append(
        f"CHR {chromosome} · {position / 1e6:.1f} MB · −LOG₁₀(P) {value:.1f}{suffix}"  # noqa: RUF001
    )
peak_display_y = np.minimum(
    neglog10_pvalue[[chr20_peak_index, other_peak_index]],
    off_scale_y,
)

genome_wide_threshold = -np.log10(5e-8)

chart = xy.scatter_chart(
    xy.scatter(
        genomic_position,
        neglog10_pvalue,
        color=point_rgba,
        size=1.35,
        opacity=0.78,
        density=True,
    ),
    xy.hline(
        genome_wide_threshold,
        color="#a33a32",
        width=2.3,
        style={"dash": "7,4"},
    ),
    xy.text(
        float(genomic_position.max()),
        genome_wide_threshold,
        "GENOME-WIDE · P = 5 \u00d7 10⁻⁸",
        dx=-10,
        dy=-20,
        color="#923a34",
        anchor="end",
        style={
            "background": "#fffdf8f2",
            "border": "1px solid #d8c7b8",
            "border_radius": 2,
            "font_size": 11.5,
            "font_weight": 750,
            "letter_spacing": "0.045em",
            "padding": "3px 6px",
        },
    ),
    xy.scatter(
        genomic_position[[chr20_peak_index, other_peak_index]],
        peak_display_y,
        color="#fffdf8",
        stroke="#81542f",
        stroke_width=2,
        size=7,
        opacity=1,
    ),
    *[
        xy.text(
            float(genomic_position[peak_index]),
            float(peak_display_y[peak_number]),
            label,
            dx=dx,
            dy=dy,
            color="#50354a",
            anchor=anchor,
            style={
                "background": "#fffdf8f2",
                "border": "1px solid #cdbfaf",
                "border_radius": 6,
                "font_size": 11.5,
                "font_weight": 700,
                "letter_spacing": "0.025em",
                "padding": "3px 6px",
            },
        )
        for peak_number, ((peak_index, dx, dy, anchor), label) in enumerate(
            zip(peak_specs, peak_labels, strict=True)
        )
    ],
    xy.x_axis(
        label=None,
        tick_label_strategy="none",
        style={
            "grid_opacity": 0,
            "axis_color": "#7c3aed66",
            "axis_width": 1.2,
            "tick_color": "#00000000",
            "tick_width": 0,
            "tick_label_color": "#00000000",
            "label_color": "#00000000",
        },
    ),
    xy.y_axis(
        label=None,
        domain=(0, display_y_max),
        tick_label_strategy="none",
        style={
            "grid_opacity": 0,
            "axis_color": "#00000000",
            "axis_width": 0,
            "tick_color": "#00000000",
            "tick_width": 0,
            "tick_label_color": "#00000000",
            "label_color": "#00000000",
        },
    ),
    xy.legend(show=False),
    xy.tooltip(
        title="Standing-height association",
        format={"x": ",.0f", "y": ".2f"},
    ),
    xy.interaction_config(
        crosshair=True,
        wheel_zoom=True,
        box_zoom=True,
        double_click_reset=True,
    ),
    xy.theme(
        background="#fffaf5",
        plot_background="#fffaf5",
        text_color="#2c2824",
        grid_color="#00000000",
        axis_color="#7c3aed66",
        crosshair_color="#a33a32",
        selection_color="#71465f",
        selection_fill="#71465f24",
    ),
    styles={
        "axis_title": {
            "font_size": 12,
            "font_weight": 650,
            "letter_spacing": "0.03em",
        },
        "tick_label": {
            "font_size": 11,
            "font_variant_numeric": "tabular-nums",
        },
        "annotation_label": {"font_weight": 650},
    },
    style={
        "font_family": "Avenir Next, ui-sans-serif, sans-serif",
    },
    width=1150,
    height=620,
    padding=(24, 32, 24, 48),
)
print(chart.memory_report()["canonical_bytes"], "canonical bytes")
chart